# Movies: what makes a film a good investment?

This notebook investigates a dataset of 3,201 films released in the US,
with box office results, production budgets, release dates, genres and
audience ratings (vega-datasets, *movies*).

**Main question:** what makes a film a good investment?
I measure this as **return on investment (ROI)**: how many dollars a film
earned for every dollar it cost to make.

**Sub-questions**
1. [Which genres give the best ROI?]
2. [Do adaptations (books, comics, remakes) pay off better than originals?]
3. [...]

**What you will find:** a first look at the data, the cleaning steps
(including impossible release dates and suspicious gross figures),
group summaries, charts, answers to the questions above, and a data note
on where the data comes from and how far it can be trusted.

**Short answer:** [write this last, 1-2 sentences]

## 0. Setup

In [81]:
# imports
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pandas.tseries.api import guess_datetime_format

# data
movies_source = pd.read_csv("data/movies/movies.csv")
genres_source = pd.read_csv("data/movies/genre_groups.csv")
movies = movies_source.copy()
genres = genres_source.copy()

# seaborn settings
sns.set_theme(style="whitegrid")

## 1. Understand the data  [task 1]

### 1.1 What is one row?

In [82]:
display(movies.sample(5, random_state=1))
display(len(movies))

,Title,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,MPAA Rating,Running Time min,Distributor,Source,Major Genre,Creative Type,Director,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes
1751,Firestorm,8123860.0,8123860.0,NaN,19000000.0,Jan 09 1998,R,NaN,20th Century Fox,Original Screenplay,Action,Contemporary Fiction,NaN,NaN,4.4,2118.0
909,Star Trek: Generations,75671262.0,120000000.0,NaN,38000000.0,Nov 18 1994,PG,NaN,Paramount Pictures,Based on TV,Adventure,Science Fiction,NaN,NaN,6.5,26465.0
979,Trois,1161843.0,1161843.0,NaN,200000.0,Feb 11 2000,NC-17,NaN,Rainforest Productions,Original Screenplay,Thriller/Suspense,Contemporary Fiction,NaN,NaN,3.3,360.0
329,Fled,17192205.0,19892205.0,NaN,30000000.0,Jul 19 1996,R,98.0,MGM,Original Screenplay,Action,Contemporary Fiction,NaN,14.0,4.9,4215.0
1885,Ghost World,6217849.0,8764007.0,NaN,5500000.0,Jul 20 2001,R,NaN,MGM,Based on Comic/Graphic Novel,Comedy,Contemporary Fiction,Terry Zwigoff,92.0,7.7,42973.0


3201

Each row is one film (3,201 in total). For every film the dataset holds:
- **Money:** US and worldwide gross, US DVD sales, production budget
- **Release info:** release date, distributor, MPAA rating, running time
- **Content:** major genre, creative type, source, director
- **Reception:** Rotten Tomatoes rating (critic or audience score, to be checked), IMDB rating and votes

### 1.2 Size of the data

In [83]:
display(movies.shape)

(3201, 16)

- The table has 3,201 rows and 16 columns.

### 1.3 Columns and data types

In [84]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 3201 entries, 0 to 3200
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Title                   3200 non-null   str    
 1   US Gross                3194 non-null   float64
 2   Worldwide Gross         3194 non-null   float64
 3   US DVD Sales            564 non-null    float64
 4   Production Budget       3200 non-null   float64
 5   Release Date            3201 non-null   str    
 6   MPAA Rating             2596 non-null   str    
 7   Running Time min        1209 non-null   float64
 8   Distributor             2969 non-null   str    
 9   Source                  2836 non-null   str    
 10  Major Genre             2926 non-null   str    
 11  Creative Type           2755 non-null   str    
 12  Director                1870 non-null   str    
 13  Rotten Tomatoes Rating  2321 non-null   float64
 14  IMDB Rating             2988 non-null   float64
 15

- Release date is stored as type string, but should be datetime64(converted in 2.1)
  
- US DVD Sales (564 of 3,201), Running Time (1,209) and Director (1,870) are missing for many films (see section 3).

- Currency is not documented. US Gross, MPAA, US DVD Sales and the Titles in english points to it being American so US dollars

### 1.4 First and last rows

In [85]:
display(movies.head())
display(movies.tail())


,Title,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,MPAA Rating,Running Time min,Distributor,Source,Major Genre,Creative Type,Director,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes
0,The Land Girls,146083.0,146083.0,NaN,8000000.0,Jun 12 1998,R,NaN,Gramercy,NaN,NaN,NaN,NaN,NaN,6.1,1071.0
1,"First Love, Last Rites",10876.0,10876.0,NaN,300000.0,Aug 07 1998,R,NaN,Strand,NaN,Drama,NaN,NaN,NaN,6.9,207.0
2,I Married a Strange Person,203134.0,203134.0,NaN,250000.0,Aug 28 1998,NaN,NaN,Lionsgate,NaN,Comedy,NaN,NaN,NaN,6.8,865.0
3,Let's Talk About Sex,373615.0,373615.0,NaN,300000.0,Sep 11 1998,NaN,NaN,Fine Line,NaN,Comedy,NaN,NaN,13.0,NaN,NaN
4,Slam,1009819.0,1087521.0,NaN,1000000.0,Oct 09 1998,R,NaN,Trimark,Original Screenplay,Drama,Contemporary Fiction,NaN,62.0,3.4,165.0


,Title,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,MPAA Rating,Running Time min,Distributor,Source,Major Genre,Creative Type,Director,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes
3196,Zack and Miri Make a Porno,31452765.0,36851125.0,21240321.0,24000000.0,Oct 31 2008,R,101.0,Weinstein Co.,Original Screenplay,Comedy,Contemporary Fiction,Kevin Smith,65.0,7.0,55687.0
3197,Zodiac,33080084.0,83080084.0,20983030.0,85000000.0,Mar 02 2007,R,157.0,Paramount Pictures,Based on Book/Short Story,Thriller/Suspense,Dramatization,David Fincher,89.0,NaN,NaN
3198,Zoom,11989328.0,12506188.0,6679409.0,35000000.0,Aug 11 2006,PG,NaN,Sony Pictures,Based on Comic/Graphic Novel,Adventure,Super Hero,Peter Hewitt,3.0,3.4,7424.0
3199,The Legend of Zorro,45575336.0,141475336.0,NaN,80000000.0,Oct 28 2005,PG,129.0,Sony Pictures,Remake,Adventure,Historical Fiction,Martin Campbell,26.0,5.7,21161.0
3200,The Mask of Zorro,93828745.0,233700000.0,NaN,65000000.0,Jul 17 1998,PG-13,136.0,Sony Pictures,Remake,Adventure,Historical Fiction,Martin Campbell,82.0,6.7,4789.0


- Release Date format is month day year(%b %d %Y)
 
- Why are US and worldwide gross identical for some films? See 2.3. Possible reason: Title had no worldwide release or US Gross value was copied over.
  

### 1.5 Number summary

In [86]:
movies.describe().round(1)

,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Running Time min,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes
count,3194.0,3.194000e+03,564.0,3200.0,1209.0,2321.0,2988.0,2988.0
mean,44002085.2,8.534340e+07,34901546.8,31069171.4,110.2,54.3,6.3,29908.6
std,62555311.4,1.499473e+08,45895121.6,35585913.4,20.2,28.1,1.3,44937.6
min,0.0,0.000000e+00,618454.0,218.0,46.0,1.0,1.4,18.0
25%,5493221.2,8.031285e+06,9906210.8,6575000.0,95.0,30.0,5.6,4828.5
50%,22019465.5,3.116893e+07,20331557.5,20000000.0,107.0,55.0,6.4,15106.0
75%,56091761.5,9.728380e+07,37794215.8,42000000.0,121.0,80.0,7.2,35810.5
max,760167650.0,2.767891e+09,352582053.0,300000000.0,222.0,100.0,9.2,519541.0


- US and Worldwide gross has a minimum value of 0.0(checked in 2.3).

- US Gross has a mean of about 44M but a median of about 22M. ""The mean is about twice the median, and the max is far above the 75th percentile. This suggests a few very high-grossing films pull the mean up (examined in section 7)."

- Production Budget has a minimum value of 218(checked in 2.3)


### 1.6 First look at genre_groups

In [87]:
display(genres.shape)
display(genres)
genres.info()

(12, 2)

,Major Genre,Genre group
0,Drama,Drama
1,Comedy,Comedy & music
2,Romantic Comedy,Comedy & music
3,Black Comedy,Comedy & music
4,Musical,Comedy & music
5,Action,Action & adventure
6,Adventure,Action & adventure
7,Western,Action & adventure
8,Thriller/Suspense,Thriller & horror
9,Horror,Thriller & horror


<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Major Genre  12 non-null     str  
 1   Genre group  12 non-null     str  
dtypes: str(2)
memory usage: 324.0 bytes


- 12 rows and 2 columns

## 2. Clean the data  [task 2]

### 2.1 Release Date: convert to datetime
Release Date is stored as text, so sorting it sorts alphabetically (Apr, Aug, Dec ...), not by date.

In [88]:
display(movies.sort_values("Release Date")[["Title", "Release Date"]].head())
display(movies.sort_values("Release Date")[["Title", "Release Date"]].tail())

,Title,Release Date
876,The Sound of Music,Apr 01 1965
266,Death Race 2000,Apr 01 1975
583,My Beautiful Laundrette,Apr 01 1986
106,"Bright Lights, Big City",Apr 01 1988
117,Bang,Apr 01 1996


,Title,Release Date
2317,MirrorMask,Sep 30 2005
1783,Serenity,Sep 30 2005
92,My Big Fat Independent Movie,Sep 30 2005
2943,The Greatest Game Ever Played,Sep 30 2005
2585,The Queen,Sep 30 2006


*TODO: what we see*

### 2.2 Release Date: impossible years and Dec 31 placeholders

*TODO: what we see*

### 2.3 Third problem (duplicates? odd values? text columns?)

*TODO: what we see*

## 3. Missing values  [task 3]

### 3.1 Count missing values per column

*TODO: what we see*

### 3.2 Decide: drop, fill or keep (and why)

*TODO: what we see*

## 4. Merge with genre groups  [task 4]

### 4.1 Row count before the merge

*TODO: what we see*

### 4.2 Merge on Major Genre

*TODO: what we see*

### 4.3 Row count after, and check nothing got lost

*TODO: what we see*

## 5. New columns  [task 5]

### 5.1 Release year

*TODO: what we see*

### 5.2 ROI (worldwide gross ÷ production budget)

*TODO: what we see*

## 6. Group summaries  [task 6]

### 6.1 Summary 1

*TODO: what we see*

### 6.2 Summary 2

*TODO: what we see*

### 6.3 Summary 3

*TODO: what we see*

## 7. A closer look at one column  [task 7]

### 7.1 Mean vs median

*TODO: what we see*

### 7.2 Spread and outliers

*TODO: what we see*

## 8. Charts  [task 8]

### 8.1 Chart 1

*TODO: what we see*

### 8.2 Chart 2

*TODO: what we see*

### 8.3 Chart 3

*TODO: what we see*

### 8.4 Chart 4

*TODO: what we see*

## 9. My questions  [task 9]

### Q1 ...

*TODO: what we see*

### Q2 ...

*TODO: what we see*

### Q3 ...

*TODO: what we see*

## 10. Data note  [task 10]

### 10.1 Source

*TODO: what we see*

### 10.2 Spot-checks against other sources

*TODO: what we see*

### 10.3 Limitations

*TODO: what we see*

## Conclusion
*TODO: 3-5 bullet answers to the main question*